# Feature Selection

*Which features earn a place in the model?* §2's on-demand review answers this in two steps: **grouped permutation importance** ranks each feature against a synthetic noise floor (§3), then a **paired-bootstrap test** refits the model on the surviving reduced set and checks whether dropping the rest actually costs PR-AUC (§4) — that second test is what decides the committed set, not the first.

`features/schema.py::COMMITTED_FEATURES` is a fixed, hand-maintained constant — re-running §2's review (on a real trigger only: a new feature, a drift signal, a scheduled review, never automatically) is the only thing that can change it. §3–5 below render **the current experiment run's own results** — nothing downstream has anything to show unless it actually ran this session. There's no cheaper fallback to browse; routine per-cycle diagnostics live in the MLflow UI directly.

In [1]:
%matplotlib inline

from __future__ import annotations

import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()

import os

os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

from telco_churn.data.eda import encoded_correlation_matrix
from telco_churn.data.split import partition
from telco_churn.features.accessor import load_features
from telco_churn.features.build import (
    COMMITTED_FEATURES,
    COMMITTED_FEATURES_DECISION,
    COMMITTED_FEATURES_DECISION_RUN_ID,
    FEATURE_SCHEMA,
    TARGET_COL,
)
from telco_churn.models.train.feature_selection import run_feature_selection_step
from telco_churn.utils.logging import configure_logging
from telco_churn.utils.paths import compose_config, get_project_root

configure_logging()
sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

## 1. Load dev partition

In [2]:
df = load_features()
dev_df, _test_df = partition(df)

# Full FEATURE_SCHEMA candidate space (not just COMMITTED_FEATURES).
_candidate_cols = list(FEATURE_SCHEMA.binary) + list(FEATURE_SCHEMA.multi_cat) + list(FEATURE_SCHEMA.numeric)
X_dev, y_dev = dev_df[_candidate_cols], dev_df[TARGET_COL]

print(f"Dev partition: {len(y_dev)} rows, churn rate {y_dev.mean():.4f}")


Dev partition: 5634 rows, churn rate 0.2654


## 2. Run the feature-set decision (on-demand review)

**Run this only on a real trigger** — a new feature, a drift signal, or a scheduled review — never as part of a routine pass. The cell below (`run_feature_selection_step`, `models/train/feature_selection.py`) recomputes the full paired-bootstrap ablation from scratch against current dev data, logging to its own `telco-churn-feature-selection-review` experiment, kept separate from the routine `telco-churn-training` runs.

**⚠ Expensive — ~85,000 scoring passes, several minutes.** If the conclusion changes, the cell prints a ready-to-paste update for `COMMITTED_FEATURES`/`COMMITTED_FEATURES_DECISION`/`COMMITTED_FEATURES_DECISION_RUN_ID` in `src/telco_churn/features/schema.py` — edit it directly via a reviewed PR.

In [3]:
RUN_ON_DEMAND_REVIEW = False  # flip to True only when a trigger from the cell above applies


In [4]:
cfg = compose_config()

tracking_uri = str(cfg.mlflow.tracking_uri)
if "://" not in tracking_uri:
    tracking_uri = str(get_project_root() / tracking_uri)
mlflow.set_tracking_uri(tracking_uri)

if RUN_ON_DEMAND_REVIEW:
    review = run_feature_selection_step(X_dev, y_dev, cfg)
    client = mlflow.tracking.MlflowClient()

    print(f"Decision: {review['decision'].upper()} ({review['decision_rule']})")

    # COMMITTED_FEATURES' declared order (binary, then multi_cat, then numeric) — used
    # by the paste block below.
    candidate_order = list(FEATURE_SCHEMA.binary) + list(FEATURE_SCHEMA.multi_cat) + list(FEATURE_SCHEMA.numeric)
    recommended = set(review["recommended_committed_features"])
    recommended_ordered = [f for f in candidate_order if f in recommended]
    today_feature_decision = review["decision"]

    # Compared on decision + membership, not run_id (a fresh run never matches an old
    # one). COMMITTED_FEATURES_DECISION_RUN_ID == 'N/A' means no decision exists yet.
    no_frozen_decision_yet = COMMITTED_FEATURES_DECISION_RUN_ID == "N/A"
    reconfirmed = (
        not no_frozen_decision_yet
        and today_feature_decision == COMMITTED_FEATURES_DECISION
        and set(recommended_ordered) == set(COMMITTED_FEATURES)
    )

    if no_frozen_decision_yet:
        print(
            "\nNo frozen decision exists yet (COMMITTED_FEATURES_DECISION_RUN_ID is "
            "'N/A') — this run establishes it."
        )
    elif reconfirmed:
        print(
            f"\nReconfirmed {COMMITTED_FEATURES_DECISION!r} — no change needed. Frozen "
            "provenance still cites COMMITTED_FEATURES_DECISION_RUN_ID="
            f"{COMMITTED_FEATURES_DECISION_RUN_ID}; today's run ({review['run_id']}) is "
            "fresh confirming evidence only."
        )
    else:
        print(
            f"\nRECOMMENDATION CHANGED: today recommends {today_feature_decision!r} "
            f"({len(recommended_ordered)} features) — differs from committed "
            f"{COMMITTED_FEATURES_DECISION!r} ({len(COMMITTED_FEATURES)} features, "
            f"run {COMMITTED_FEATURES_DECISION_RUN_ID})."
        )

    if not reconfirmed:
        # No diff to show in the bootstrap case — nothing "current" to compare against.
        if not no_frozen_decision_yet:
            dropped = [f for f in COMMITTED_FEATURES if f not in recommended]
            added = [f for f in recommended_ordered if f not in set(COMMITTED_FEATURES)]
            if dropped:
                print(f"\nDropped vs. the current COMMITTED_FEATURES ({len(dropped)}):")
                for f in dropped:
                    print(f"  - {f}")
            if added:
                print(f"\nAdded vs. the current COMMITTED_FEATURES ({len(added)}):")
                for f in added:
                    print(f"  + {f}")

        paste_block = (
            f'COMMITTED_FEATURES_DECISION: str = "{today_feature_decision}"\n'
            f'COMMITTED_FEATURES_DECISION_RUN_ID: str = "{review["run_id"]}"\n'
            "COMMITTED_FEATURES: tuple[str, ...] = (\n"
            + "\n".join(f'    "{f}",' for f in recommended_ordered)
            + "\n)"
        )
        print(
            "\nTo update src/telco_churn/features/schema.py: replace "
            "COMMITTED_FEATURES_DECISION, COMMITTED_FEATURES_DECISION_RUN_ID, and "
            "COMMITTED_FEATURES with the block below (each feature survived at least "
            "half of run_selection_cv's 100 folds — stability >= 0.5 — not a single "
            "all-dev fit's fluke-prone call; also logged as this run's "
            "recommended_committed_features.txt), and record the rationale in "
            "ANALYSIS.md §4b — via a reviewed PR, never a direct push.\n"
        )
        print(paste_block)

    print(
        "\nSection 3 onward downloads and renders this run's own artifacts from MLflow "
        f"(run_id={review['run_id']}) — not a separate, previously-logged review run."
    )
else:
    review = None
    print(
        "Skipped — RUN_ON_DEMAND_REVIEW is False. Flip it above only when a real trigger "
        "applies (see the cell above); routine execution of this notebook does not need it. "
        "Sections 3 onward have nothing to render without it."
    )


Skipped — RUN_ON_DEMAND_REVIEW is False. Flip it above only when a real trigger applies (see the cell above); routine execution of this notebook does not need it. Sections 3 onward have nothing to render without it.


## 3. Permutation-importance ranking

**Which features carry real signal on their own?** Each feature's one-hot dummies are shuffled together and the resulting PR-AUC drop is measured against a synthetic **noise column** built to hold no information — a feature "survives" only if it clears that noise floor (the dashed line in the chart below) by at least `noise_floor_margin = 0.005` on a majority of the review's 100 cross-validated folds (**stability ≥ 0.5**), not just a single fit. This is model-agnostic, unlike LightGBM's own built-in **gain** importance (how often a feature is used to split trees). The table and chart below sort features by that PR-AUC drop — green rows/bars cleared the floor, red ones didn't.

In [5]:
if review is not None:
    import base64

    from IPython.display import HTML, display

    # Targeted downloads (mirrors 03a's comp_dir/diag_dir pattern) rather than
    # the whole run tree: only the files this notebook actually reads, each
    # call's own thread burst staying well under urllib3's connection-pool cap.
    figures_dir = mlflow.artifacts.download_artifacts(
        run_id=review["run_id"], artifact_path="figures"
    )
    perm_table_path = mlflow.artifacts.download_artifacts(
        run_id=review["run_id"], artifact_path="permutation_importance_table.csv"
    )
    importance_table = pd.read_csv(perm_table_path).sort_values(
        "real_importance", ascending=False
    )

    decoy_importance = float(importance_table["decoy_importance"].iloc[0])
    print(f"Decoy column importance (mean across the review's 100 ablation folds): {max(decoy_importance, 0.0):.4f}\n")

    n_survived = int(importance_table["survived"].sum())
    n_total = len(importance_table)
    print(
        f"{n_survived} of {n_total} features cleared the decoy floor on at least half of "
        "the review's 100 folds (stability >= 0.5).\n"
    )

    def _highlight(row):
        style = (
            "background-color: #c6efce; color: #006100"
            if row["survived"]
            else "background-color: #ffc7ce; color: #9c0006"
        )
        return [style] * len(row)

    display_table = importance_table[["feature", "real_importance", "importance_floor", "survived"]]
    table_html = (
        display_table.style.apply(_highlight, axis=1)
        .format({"real_importance": "{:.4f}", "importance_floor": "{:.4f}"})
        .hide(axis="index")
        .to_html()
    )

    # Pre-rendered by run_feature_selection_step itself (figures/permutation_importance.png)
    # — downloaded here, not rebuilt.
    with open(f"{figures_dir}/permutation_importance.png", "rb") as f:
        chart_b64 = base64.b64encode(f.read()).decode("ascii")

    display(HTML(
        '<div style="display:flex; align-items:flex-start; gap:24px; flex-wrap:wrap;">'
        f'<div style="flex:0 1 auto; min-width:320px; overflow:auto;">{table_html}</div>'
        f'<div style="flex:0 1 auto; min-width:320px;"><img src="data:image/png;base64,{chart_b64}" style="max-width:100%;"></div>'
        "</div>"
    ))
else:
    print("Skipped — the review above did not run this session.")


Skipped — the review above did not run this session.


### Per-fold stability

If the selector is refit across every outer CV fold (`features.select.run_selection_cv`, `cv_folds x cv_repeats` = 10x10 = 100 fits) — does it keep choosing the same features, or does the survivor list bounce around fold to fold? Each fold's selector refits from scratch on that fold's own training portion, never seeing its held-out rows, so the check stays leak-free. The chart below shows, per feature: position = the fraction of the 100 folds it survived on, colour = whether that fraction clears the same stability ≥ 0.5 threshold behind the `survived` column above.

In [6]:
if review is not None:
    from IPython.display import Image, display

    display(Image(filename=f"{figures_dir}/per_fold_stability.png"))
else:
    print("Skipped — the review above did not run this session.")


Skipped — the review above did not run this session.


*Reflects the review's last actual run this session.* The per-fold vote mostly confirms the survivor list, but not uniformly: `tenure`/`contract_type` clear every one of the 100 folds, and five more survivors (`totalcharges`, `monthlycharges`, `internetservice`, `techsupport`, `onlinesecurity`) sit at roughly 75–95%. The two thinnest survivors are genuinely marginal, though — `charge_per_service` at ~69/100 and `paperlessbilling` at just ~51/100, barely above the 50% cutoff that decides `survived`. Two non-survivors sit closer to that line than the rest of the pack: `paymentmethod` at ~38/100 and `streamingtv` at ~8/100 — both clear minorities of folds, but `paymentmethod` in particular isn't that far below where `paperlessbilling` sits above. This is exactly why the aggregate paired-bootstrap test below (§4) — not any single feature's threshold call — is what actually decides `COMMITTED_FEATURES`: at this margin, a few points of fold-stability either way isn't a result to hang a keep/drop decision on for any one feature.

### Correlated feature groups

When features are highly correlated, permutation importance can split credit unevenly across them — starving one member below the decoy floor even though the group as a whole carries real signal. Two such groups exist in this feature set (per the correlation heatmaps below and the VIF check, §8a/§8b `01-eda.ipynb`): before accepting any individual failure, a group-level rescue check re-tests each cluster together — but only when every member of that cluster failed on its own (see the shared lookup below).

A group's rescue only ever changes *which* features a `'reduced'` outcome would keep — never *whether* `'reduced'` wins over `'full'` in the first place; that vote is §4's alone.

In [7]:
correlated_groups = [list(group) for group in cfg.selection.correlated_groups]
print("Correlated groups checked:")
print(f"  - tenure / totalcharges / monthlycharges: {correlated_groups[0]}")
print(f"  - Internet-service add-on cluster: {correlated_groups[1]}")

if review is not None:
    import json

    group_importance_path = mlflow.artifacts.download_artifacts(
        run_id=review["run_id"], artifact_path="group_importance.json"
    )
    with open(group_importance_path) as f:
        group_importance = json.load(f)


Correlated groups checked:
  - tenure / totalcharges / monthlycharges: ['tenure', 'totalcharges', 'monthlycharges']
  - Internet-service add-on cluster: ['internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies']


#### `tenure` / `totalcharges` / `monthlycharges`

`totalcharges` accrues from `monthlycharges` over `tenure` — the clearest linear dependency in the dataset.

Spearman correlation.

In [8]:
if review is not None:
    trio = correlated_groups[0]
    corr = dev_df[trio].corr(method="spearman")

    fig, ax = plt.subplots(figsize=(5, 4.5))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
    ax.set_title("Spearman correlation — tenure / totalcharges / monthlycharges")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — the review above did not run this session.")


Skipped — the review above did not run this session.


Each member's own permutation-importance read, individually.

In [9]:
if review is not None:
    from IPython.display import display

    display(importance_table[importance_table["feature"].isin(trio)][
        ["feature", "real_importance", "importance_floor", "survived"]
    ])
else:
    print("Skipped — the review above did not run this session.")


Skipped — the review above did not run this session.


The group-level rescue check — populated only if every member above failed individually.

In [10]:
if review is not None:
    trio_key = "|".join(trio)
    if trio_key in group_importance:
        print(
            f"Joint permutation importance of {trio}: {group_importance[trio_key]:.5f} "
            f"(floor: {float(importance_table['importance_floor'].iloc[0]):.4f})"
        )
    else:
        print(
            f"{trio} is not in group_importance.json — the rescue only runs when every "
            "member fails the individual floor; at least one must have survived alone "
            "across the review's 100 folds."
        )
else:
    print("Skipped — the review above did not run this session.")


Skipped — the review above did not run this session.


*Reflects the review's last actual run this session.* All three clear the floor alone despite `tenure`/`totalcharges` correlating at 0.89 — the rescue never needs to fire. But the importance gap is wide: `tenure` (0.103) scores 3-4x `totalcharges` (0.030) and `monthlycharges` (0.023), so even though `totalcharges` accrues almost mechanically from `monthlycharges` over `tenure`, the model leans on `tenure` itself far more than on either component of that accrual. `totalcharges`/`monthlycharges` still clear the floor several times over (0.030/0.023 vs. 0.008) though — real, independent signal, just a smaller share of it.

#### Internet-service add-on cluster

`internetservice` and its six add-on services (`onlinesecurity`, `onlinebackup`, `deviceprotection`, `techsupport`, `streamingtv`, `streamingmovies`) are the most collinear block in the dataset: `internetservice_No` propagates identically across all six add-on columns (VIF = ∞, §8b `01-eda.ipynb`) whenever a customer has no internet service — a structural floor shared by all seven columns simultaneously.

In [11]:
if review is not None:
    internet_group = correlated_groups[1]
    internet_corr = encoded_correlation_matrix(dev_df[internet_group], cat_cols=internet_group)

    mask = np.triu(np.ones_like(internet_corr, dtype=bool))
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(
        internet_corr, mask=mask, cmap="coolwarm", center=0,
        vmin=-1, vmax=1, annot=False, linewidths=0.3, ax=ax
    )
    ax.set_title("Correlation (one-hot) — internet-service add-on cluster")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — the review above did not run this session.")


Skipped — the review above did not run this session.


Each member's own permutation-importance read, individually.

In [12]:
if review is not None:
    from IPython.display import display

    display(importance_table[importance_table["feature"].isin(internet_group)][
        ["feature", "real_importance", "importance_floor", "survived"]
    ])
else:
    print("Skipped — the review above did not run this session.")


Skipped — the review above did not run this session.


The group-level rescue check — populated only if every member above failed individually.

In [13]:
if review is not None:
    internet_key = "|".join(internet_group)
    if internet_key in group_importance:
        print(
            f"Joint permutation importance of {internet_group}: "
            f"{group_importance[internet_key]:.5f} "
            f"(floor: {float(importance_table['importance_floor'].iloc[0]):.4f})"
        )
    else:
        print(
            f"{internet_group} is not in group_importance.json — the rescue only runs "
            "when every member fails the individual floor; at least one must have "
            "survived alone across the review's 100 folds."
        )
else:
    print("Skipped — the review above did not run this session.")


Skipped — the review above did not run this session.


*Reflects the review's last actual run this session.* Only 3 of the 7 members clear the floor alone (`internetservice`, `techsupport`, `onlinesecurity`); the other 4 read as noise individually. But the group-level rescue fires here — joint importance 0.062, ~8x the floor and well above the three survivors' combined score (0.049) — so the four "failed" members do carry real signal, it's just invisible one at a time. Why: the heatmap above shows the `*_No internet service` columns are structurally almost perfectly correlated with each other (every add-on reads "No internet service" whenever a customer has no internet) — shuffle one and the model reconstructs its contribution from the rest; shuffle all seven together and it can't. None of this changes what ships — `COMMITTED_FEATURES` includes the full cluster regardless (§4 below).

### Fairness check: demographic block

This is **not** a correlation-dilution check like the two groups above — `seniorcitizen`'s VIF is ≈1.15 (essentially independent of everything else, §8b `01-eda.ipynb`), so these four aren't meaningfully correlated with each other or with anything else — no correlation matrix to show here for that reason. The question here is different: **how much does this whole category of protected/quasi-protected attributes collectively buy the model**, a number a fairness-motivated removal decision would need — not whether correlation is hiding a real signal that individual permutation importance under-counts. Individually weak by construction (Cramér's V 0.15–0.16 for three of the four, per §7 `01-eda.ipynb`) — none is expected to clear the floor alone.

Each member's own permutation-importance read, individually — all four expected to fail (see above).

In [14]:
if review is not None:
    from IPython.display import display

    demo_group = correlated_groups[2]
    display(importance_table[importance_table["feature"].isin(demo_group)][
        ["feature", "real_importance", "importance_floor", "survived"]
    ])
else:
    print("Skipped — the review above did not run this session.")


Skipped — the review above did not run this session.


The group-level rescue check that actually answers the fairness-pricing question above.

In [15]:
if review is not None:
    demo_key = "|".join(demo_group)
    if demo_key in group_importance:
        print(
            f"Joint permutation importance of {demo_group}: "
            f"{group_importance[demo_key]:.5f} "
            f"(floor: {float(importance_table['importance_floor'].iloc[0]):.4f})"
        )
    else:
        print(
            f"{demo_group} is not in group_importance.json — the rescue only runs when "
            "every member fails the individual floor; at least one must have survived "
            "alone across the review's 100 folds."
        )
else:
    print("Skipped — the review above did not run this session.")


Skipped — the review above did not run this session.


*Reflects the review's last actual run this session.* This whole category of protected/quasi-protected attributes collectively buys the model at most ~0.003 PR-AUC. All four members fail the individual floor alone (0.001, 0.001, 0.001, −0.001 vs. a 0.008 floor), which is exactly the rescue's trigger condition, so the joint permutation runs and returns `group_importance = 0.00309`, still comfortably below the floor. That's the number a fairness-motivated removal of this block would need to weigh against — cheap to give up, on this evidence — and it's one marginal importance alone could never produce, since none of the four carries enough signal individually to measure.

## 4. Full-set vs. reduced-set paired-bootstrap test

Does dropping to the smaller, permutation-importance-surviving feature set actually cost PR-AUC, or is the leaner model just as good? Tested with a **paired bootstrap** on Δ = mean(AP_full) − mean(AP_reduced): each CV fold's full-feature score is paired against that same fold's reduced-feature score (identical `RepeatedStratifiedKFold` instance), which cancels intra-fold variance — more power than checking whether the reduced mean merely falls inside the full model's own *unpaired* CI. Same mechanism as the model-family decision (§3 `03a-model-selection.ipynb`).

This is a single **aggregate** test of the whole reduced set at once, refit and cross-validated by `run_selection_cv` — it asks whether dropping *all* of §3's non-survivors together costs real PR-AUC, not whether any one specific feature's removal would. `reduced_set_bootstrap_test` is what actually decides `COMMITTED_FEATURES` — not §3's shuffle-based, no-refit scores.

**Pre-registered decision rule** (materiality threshold Δ\* = 0.005 PR-AUC — a real gap can still be too small to act on). The reduced set is the default: a leaner model wins unless the evidence says otherwise.

| Branch | Condition | Plain English | Outcome |
|---|---|---|---|
| `full_features_win` | CI excludes 0 in full's favour AND Δ ≥ Δ\* | Dropping features costs real PR-AUC | Adopt the full set |
| `tie` | CI includes 0, or excludes 0 but \|Δ\| < Δ\* | Can't tell them apart, or the gap is too small to matter | Adopt the reduced set (the simpler default) |
| `reduced_features_win` | CI excludes 0 in reduced's favour AND Δ ≤ −Δ\* | The reduced set is actually better | Adopt the reduced set on the evidence |

In [16]:
if review is not None:
    run_data = client.get_run(review["run_id"]).data

    full_pr_auc = run_data.metrics["full_cv_pr_auc_mean"]
    reduced_pr_auc = run_data.metrics["reduced_cv_pr_auc_mean"]
    delta_obs = run_data.metrics["bootstrap_delta_obs"]
    ci_lo = run_data.metrics["bootstrap_delta_ci_lower"]
    ci_hi = run_data.metrics["bootstrap_delta_ci_upper"]
    p_value = run_data.metrics["bootstrap_p_value"]
    fold_win_rate = run_data.metrics["fold_win_rate"]
    delta_threshold = float(run_data.params["delta_threshold"])

    print(f"Full-feature CV PR-AUC    : {full_pr_auc:.4f}  (mean of per-fold APs)")
    print(f"Reduced-feature CV PR-AUC : {reduced_pr_auc:.4f}  (mean of per-fold APs)")
    print(f"Paired Δ (full - reduced) : {delta_obs:+.4f}, 95% CI [{ci_lo:+.4f}, {ci_hi:+.4f}], p={p_value:.4f}")
    print(f"Δ*                        : {delta_threshold:.4f}")
    print(f"Decision                  : {today_feature_decision}  ({review['decision_rule']})")
    print(f"Fold win rate             : full beats reduced on {fold_win_rate:.0%} of folds")

    import matplotlib.image as mpimg

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].imshow(mpimg.imread(f"{figures_dir}/pr_curves.png"))
    axes[0].axis("off")
    axes[1].imshow(mpimg.imread(f"{figures_dir}/bootstrap_delta_dist.png"))
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — the review above did not run this session.")


Skipped — the review above did not run this session.


*Reflects the review's last actual run this session.* The CI [+0.0050, +0.0120] excludes zero entirely in the full set's favour, and the observed gap (Δ = 0.0080) clears the materiality threshold (Δ\* = 0.005) — so `full_features_win` fires and the full set is adopted. This is a decisive win, not a coin flip: full beats reduced on 65 of 100 CV folds, so the same small edge repeats fold after fold rather than washing out under averaging.

The two plots above use different, separately-logged metrics, not two readings of the same number: the PR curves (left, "pooled OOF AP") pool every fold's predictions before scoring once, which smooths away the fold-to-fold edge — both curves land at AP = 0.66 to two decimal places, indistinguishable at a glance. The bootstrap plot (right, "Δ mean-of-fold PR-AUC") scores each fold first and pairs full vs. reduced within that same fold before averaging, so the same repeating edge accumulates instead of washing out (0.6583 vs. 0.6499, Δ = 0.0080) — that's the version the decision rule actually uses.

This is the test §3's per-fold stability check flagged as the actual arbiter for the borderline features — the two thinnest survivors (`charge_per_service` at ~69/100 folds, `paperlessbilling` at ~51/100) and the two closest-to-the-line non-survivors (`paymentmethod` at ~38/100, `streamingtv` at ~8/100). It resolves those marginal calls directly: none of the four is hand-picked out individually here — the full 20-feature set is adopted as a block, so all four ship (or don't) together with everything else, regardless of how close each sat to its own 50% line.

### Reasons for keeping the failed features

The bootstrap test above settled on the full set, so none of the 10 features failing their own permutation-importance floor (§3) are dropped. Full narrative and the historically-decided numbers: [§4b `ANALYSIS.md`](../ANALYSIS.md#4b-feature-selection-concluded-ablation--importance-diagnostic), MLflow run `b918a87ef6e3425f992ccd1f7c714a1b` (predates the Phase 12a AWS migration, logged against local dev's backend — not reachable from the RDS+S3-backed reviewer-facing MLflow UI; run `146379a23a4642aca7fc1692a13aa7bb`, 2026-09-18, reconfirmed the same decision against RDS+S3 and is the AWS-reachable run instead).

Per §7 `01-eda.ipynb`, most of the 10 features failing their own permutation-importance floor above do have real, statistically significant univariate correlation with churn — the add-on cluster `onlinebackup`/`deviceprotection`/`streamingmovies`/`streamingtv` (Cramér's V 0.23–0.29) and the protected-attribute trio `seniorcitizen`/`has_partner`/`dependents` (V = 0.15–0.16). `paymentmethod` is the sharpest case: at V = 0.30 it's one of the strongest univariate correlates in the dataset — stronger than everything in the add-on cluster except `onlinebackup` — yet it contributes essentially nothing once the other 19 features are already in the model. That signal just doesn't survive as *marginal* contribution — but it isn't a simple "redundant with one bigger feature" story throughout: the add-on cluster's own rescue check above found the opposite of redundancy — real credit-splitting, with a joint importance ~8x the floor and above even the surviving members' combined score — so those four failing individually is exactly what that check exists to catch. `seniorcitizen` is different again — it barely correlates with any other feature in the dataset at all (a formal collinearity check gives it VIF ≈ 1.15, essentially independent — §8b `01-eda.ipynb`), so it isn't being crowded out by an overlapping feature either. `gender` and `phoneservice` are the cleanest case: EDA also found no real association for either (V ≈ 0.01, both fail to reject the null), so both tests agree there's no signal. `multiplelines` runs the other way — weak in EDA (V = 0.04), but still clears permutation importance here.

So why do any of these 10 stay in the model at all — including `gender` and `phoneservice`, which carry no signal by either measure? Because the decision above is a single yes/no choice between the full 20-feature set and the reduced 10-feature set, not a per-feature filter. Once the full set wins that aggregate test, every one of the 20 rides along together — there's no intermediate step that drops individually-null features on their own.

## 5. SHAP audit (diagnostic only)

**Does a different way of measuring importance agree with §3?** SHAP looks at how much each feature actually moves each customer's individual prediction, then averages that across everyone — a different lens from §3's permutation importance, which instead measures the PR-AUC cost of scrambling a feature. Two charts below render the same SHAP numbers, both pre-rendered by §2's review run: first coloured by whether the feature survived §3's floor, then coloured by whether it's part of this run's recommended feature set. Diagnostic only, like §3 — neither chart decides which features ship. Both render only if §2's review ran this session.

In [17]:
if review is not None:
    import base64

    from IPython.display import HTML, display

    def _shap_chart_b64(filename: str) -> str:
        with open(f"{figures_dir}/{filename}", "rb") as f:
            return base64.b64encode(f.read()).decode("ascii")

    survived_b64 = _shap_chart_b64("shap_importance_audit_survived.png")
    committed_b64 = _shap_chart_b64("shap_importance_audit.png")

    display(HTML(
        '<div style="display:flex; align-items:flex-start; gap:24px; flex-wrap:wrap;">'
        f'<div style="flex:0 1 auto; min-width:320px;"><img src="data:image/png;base64,{survived_b64}" style="max-width:100%;"></div>'
        f'<div style="flex:0 1 auto; min-width:320px;"><img src="data:image/png;base64,{committed_b64}" style="max-width:100%;"></div>'
        "</div>"
    ))
else:
    print("Skipped — the review above did not run this session.")


Skipped — the review above did not run this session.


*Reflects the review's last actual run this session.* 9 of the 10 §3 survivors occupy 9 of the top-10 SHAP ranks. The exception is `paymentmethod` — a non-survivor here (real importance 0.006, below the 0.008 floor) — which breaks into 3rd place by SHAP (mean |SHAP| ≈0.35), pushing the 10th survivor, `paperlessbilling`, down to 11th (≈0.11). `streamingtv`, the review's other borderline non-survivor, ranks 12th — consistent with its own permutation-importance result, not a contradiction of it. `contract_type` leads both rankings this run, well ahead of `tenure` by SHAP (≈1.12 vs. ≈0.47) despite the two being almost tied by permutation importance (0.103 each). None of this changes anything downstream: both audits are diagnostic only.

## Summary

This notebook runs §2's on-demand ablation review — off by default, triggered only on a real trigger — then renders that same review's own results: the permutation-importance ranking against the decoy floor and the correlated-group rescue checks (§3), the standing rationale for why every feature ships regardless (§4), and the non-gating SHAP audit (§5). Nothing downstream has anything to render unless §2's review actually ran this session — there is no separate, routinely-refreshed fallback; the automated per-cycle diagnostic (`run_feature_audit_step`, `src/telco_churn/models/train/feature_audit.py`) still runs every training cycle and logs to MLflow, but this notebook no longer loads or renders it.

The committed feature set — `features/schema.py::COMMITTED_FEATURES` — is a fixed, hand-maintained constant, currently the full 20-feature space, settled once (§4) and never re-derived per cycle. When §2's review does run, its own cell prints the decision plus — if it flips to `'reduced'` — the exact recommended feature list, a diff against the current constant, and a ready-to-paste replacement block for `schema.py`. Full narrative: **[§4b Feature Selection (concluded ablation + importance diagnostic)](../ANALYSIS.md#4b-feature-selection-concluded-ablation--importance-diagnostic)** in `ANALYSIS.md`. Continue to **`03c-hyperparameter-tuning.ipynb`**.